# Predictive Maintenance Data Preprocessing
This notebook implements the preprocessing pipeline for the AI4I dataset.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

df = pd.read_csv('../dataset/predictive_maintenance.csv')
df = df.drop_duplicates()
print(f'Shape after dropping duplicates: {df.shape}')

In [ ]:
# Drop identifiers and leakage columns
identifiers = ['UDI', 'Product ID']
leakage_cols = ['Failure Type']
target_col = 'Target'

X = df.drop(columns=identifiers + leakage_cols + [target_col], errors='ignore')
y = df[target_col]
print('Features shape:', X.shape)
X.head()

In [ ]:
# Stratified Train, Validation, Test Split (70%, 15%, 15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

val_fraction = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_fraction, random_state=42, stratify=y_train_val
)
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'X_test shape: {X_test.shape}')

In [ ]:
# Build preprocessing pipeline
categorical_features = ['Type']
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ])

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
final_feature_names = numeric_features + list(cat_names)
print('Final features:', final_feature_names)

In [ ]:
# Save preprocessor for reuse
os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(final_feature_names, '../models/feature_names.pkl')
print('Preprocessor saved successfully.')